# Notebook 81: AMBIGUOUS-Treatment Benchmark after Notebook 79

Purpose: quantify the visual observation from Notebook 79 that disabling `AMBIGUOUS -> Telea` appears to retain materially more printed notation, at the cost of retaining more annotation.

This notebook **does not alter routing, masks, p_print, p_remove, SAFE_REMOVE, or Ring-2 geometry**. It compares five treatment arms only at pixels already routed `AMBIGUOUS` by the frozen V7.8/V8.x route construction.

## Arms

- **A0 — TELEA (incumbent treatment):** `AMBIGUOUS -> Telea`.
- **A1 — OBSERVED:** `AMBIGUOUS -> exact observed byte`.
- **A2 — PROBABILITY BLEND:** `AMBIGUOUS -> p_print * observed + (1-p_print) * Telea`. No new threshold is introduced.
- **A3 — PIXEL STRUCTURAL GATE:** preserve an AMBIGUOUS pixel when calibrated `p_print >= 0.50`; otherwise use Telea. `0.50` is fixed before truth is loaded and is diagnostic only.
- **A4 — COMPONENT GATE:** 8-connected AMBIGUOUS components are preserved whole when the component median calibrated `p_print >= 0.50`; otherwise the entire component uses Telea.

For all arms:

- `REMOVE -> frozen local paper estimate`
- `PRESERVE -> observed`
- V8.9 Ring-2 exact observed-byte preservation is applied last.
- Pixels outside the frozen dilation-2 authority remain observed.

The notebook first locks all five observable outputs. Clean truth is loaded only afterwards for the paired development cohort. HITL pages receive no truth-derived scoring and are used for whole-page visual diagnostics only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, importlib.util, sys, gc
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V89_OPERATOR_PATH=ROOT/'v8_9_release'/'v8_9_operator.py'
P_PRINT_BUNDLE=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
NB72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
CLEAN_DIR=ROOT/'source'/'clean'
ALIGNED_DIR=ROOT/'aligned'
SEED_DIRS=[ROOT/'masks'/'corrected']
MANIFEST_PATH=ROOT/'development_score_manifest.csv'

RUN=ROOT/'notebook81_ambiguous_treatment_benchmark'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook81')
for p in [RUN,RUN/'pages',RUN/'tables',RUN/'visual',RUN/'audit',RUN/'upload']:
    p.mkdir(parents=True,exist_ok=False)

ARMS=['A0_TELEA','A1_OBSERVED','A2_PPRINT_BLEND','A3_PIXEL_GATE','A4_COMPONENT_GATE']
PPRINT_GATE=0.50
EXPECTED_HITL=151


def require(x,m):
    if not bool(x): raise RuntimeError(m)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def latest(root,required):
    hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in required)]
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def image_map(folder):
    if not Path(folder).exists(): return {}
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}

def read_gray(p):
    im=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
    require(im is not None,f'Unreadable image: {p}')
    return im

def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

require(V89_OPERATOR_PATH.exists(),f'Missing {V89_OPERATOR_PATH}')
spec=importlib.util.spec_from_file_location('v8_9_operator',V89_OPERATOR_PATH)
mod=importlib.util.module_from_spec(spec);sys.modules['v8_9_operator']=mod;spec.loader.exec_module(mod)
op=mod.load_v89_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE)
require(type(op).__name__=='V89Operator',f'Expected V89Operator, got {type(op).__name__}')
print('Operator:',type(op).__name__)
print('Run:',RUN)


## Build both cohorts

HITL uses the reviewed Notebook-72 roster. Paired development uses the canonical aligned annotated page, clean page, reviewed seed and V7.5 cache. Clean pages are *registered now but are not opened* until after the observable prediction lock closes.


In [ ]:
NB72=latest(NB72_ROOT,['inputs/runtime_roster.csv'])
require(NB72 is not None,'Complete Notebook72 runtime roster not found.')
hitl=pd.read_csv(NB72/'inputs/runtime_roster.csv')
require(len(hitl)==EXPECTED_HITL,f'Expected 151 HITL pages, found {len(hitl)}')
hitl_rows=[]
for _,r in hitl.iterrows():
    hitl_rows.append({'page_key':'hitl__annotated_only_'+str(r.page),'page':str(r.page),'cohort':'HITL_ANNOTATED_ONLY','source_group':str(r.get('cohort','HITL')),'observed_path':str(r.observed_path),'seed_path':str(r.seed_mask_path),'clean_path':None})

clean_map=image_map(CLEAN_DIR);obs_map=image_map(ALIGNED_DIR)
seed_map={}
for folder in SEED_DIRS:
    seed_map.update(image_map(folder))
manifest=pd.read_csv(MANIFEST_PATH,dtype=str).fillna('')
meta=manifest.set_index('page')
paired_pages=sorted(set(clean_map)&set(obs_map)&set(seed_map)&set(meta.index.astype(str)))
paired_rows=[{'page_key':'paired__'+p,'page':p,'cohort':'PAIRED_DEVELOPMENT','source_group':str(meta.loc[p,'source_group']),'observed_path':str(obs_map[p]),'seed_path':str(seed_map[p]),'clean_path':str(clean_map[p])} for p in paired_pages]
roster=pd.DataFrame(hitl_rows+paired_rows)
roster.to_csv(RUN/'tables'/'roster.csv',index=False)
print(roster.groupby('cohort').size())


## Arm constructor

This is the key experimental isolation. Frozen route masks, frozen paper surface, frozen Telea result and frozen calibrated `p_print` are computed once. Only the treatment of already-AMBIGUOUS pixels changes. V8.9 Ring-2 preservation is then applied identically to all arms.


In [ ]:
def build_arms(observed,seed,page):
    # Run frozen V8.9 once to obtain p_print and exact Ring-2 geometry.
    v89=op.run_page(observed,seed,page=page)

    mask=mod.prepare_mask(seed,2)
    telea=mod.telea_reconstruct(observed,mask)
    baseline,routes=mod.geometry_t80(observed,mask,telea)
    preserve=np.asarray(routes['preserve'],bool)
    remove=np.asarray(routes['remove'],bool)
    ambiguous=np.asarray(routes['ambiguous'],bool)
    paper=np.asarray(routes['paper'])
    ring2=np.asarray(v89.ring2_mask,bool)
    p_print=np.asarray(v89.p_print,dtype=np.float32)

    # If p_print is unavailable at an AMBIGUOUS coordinate, probability-based
    # arms fail back to incumbent Telea rather than guessing preservation.
    pp=np.where(np.isfinite(p_print),np.clip(p_print,0.0,1.0),0.0)

    base=observed.copy()
    base[remove]=paper[remove]
    base[preserve]=observed[preserve]

    arms={}
    a0=base.copy();a0[ambiguous]=telea[ambiguous]
    a1=base.copy();a1[ambiguous]=observed[ambiguous]

    a2=base.copy()
    blend=np.rint(pp.astype(np.float32)*observed.astype(np.float32)+(1.0-pp.astype(np.float32))*telea.astype(np.float32)).clip(0,255).astype(np.uint8)
    a2[ambiguous]=blend[ambiguous]

    gate=ambiguous & (pp>=PPRINT_GATE)
    a3=base.copy();a3[ambiguous]=telea[ambiguous];a3[gate]=observed[gate]

    a4=base.copy();a4[ambiguous]=telea[ambiguous]
    n,labels=cv2.connectedComponents(ambiguous.astype(np.uint8),connectivity=8)
    component_preserve=np.zeros_like(ambiguous,bool)
    for cid in range(1,n):
        comp=labels==cid
        vals=pp[comp]
        if len(vals) and float(np.median(vals))>=PPRINT_GATE:
            component_preserve[comp]=True
    a4[component_preserve]=observed[component_preserve]

    arms['A0_TELEA']=a0;arms['A1_OBSERVED']=a1;arms['A2_PPRINT_BLEND']=a2;arms['A3_PIXEL_GATE']=a3;arms['A4_COMPONENT_GATE']=a4

    # Apply identical V8.9 Ring-2 exact preservation last.
    for k in arms:
        arms[k][ring2]=observed[ring2]
        require(np.array_equal(arms[k][mask==0],observed[mask==0]),f'{page} {k}: outside-mask drift')
        require(np.array_equal(arms[k][ring2],observed[ring2]),f'{page} {k}: Ring2 mismatch')

    return {
        'arms':arms,'mask':mask>0,'preserve':preserve,'remove':remove,'ambiguous':ambiguous,
        'paper':paper,'telea':telea,'p_print':p_print,'ring2':ring2,'pixel_gate':gate,'component_gate':component_preserve
    }


## Generate and lock all observable outputs

No clean page is opened in this cell. The outputs, route masks and hashes are frozen before truth scoring.


In [ ]:
# ============================================================
# Generate and lock all observable outputs
#
# RESUMABLE VERSION
#
# Behaviour:
#   1. Reuse complete existing page outputs.
#   2. Fail closed on partial existing page outputs.
#   3. Skip zero-authority pages before build_arms().
#   4. Skip pages with no experimental AMBIGUOUS population.
#   5. Process only genuinely new/exposed pages.
#   6. Rebuild the summary/hash/lock from BOTH reused and
#      newly generated outputs.
#
# No clean truth is read in this cell.
# ============================================================


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def completed_page_output(page_dir):
    """
    A processed page is reusable only if the complete expected
    observable-output set exists.
    """

    page_dir = Path(page_dir)

    required = [
        page_dir / "observed.png",
        page_dir / "seed.png",
        page_dir / "ambiguous.png",
        page_dir / "ring2.png",
        page_dir / "diagnostic_arrays.npz",
        *[
            page_dir / f"{arm}.png"
            for arm in ARMS
        ],
    ]

    return all(
        p.exists() and p.is_file()
        for p in required
    )


def existing_page_state(page_dir):
    """
    Distinguish:
        MISSING
        COMPLETE
        PARTIAL
    """

    page_dir = Path(page_dir)

    if not page_dir.exists():
        return "MISSING"

    if completed_page_output(page_dir):
        return "COMPLETE"

    return "PARTIAL"


def skip_record(
    key,
    page,
    cohort,
    source_group,
    status,
    reason,
    seed_pixels=0,
    ambiguous_pixels=0,
    ring2_pixels=0,
):
    """
    Standard summary row for zero-exposure pages.
    """

    return {
        "page_key":
            str(key),

        "page":
            str(page),

        "cohort":
            str(cohort),

        "source_group":
            str(source_group),

        "status":
            str(status),

        "skip_reason":
            str(reason),

        "seed_pixels":
            int(seed_pixels),

        "ambiguous_pixels":
            int(ambiguous_pixels),

        "ring2_pixels":
            int(ring2_pixels),

        "pixel_gate_pixels":
            0,

        "component_gate_pixels":
            0,

        **{
            f"{arm}_changed_vs_observed": 0
            for arm in ARMS
        },
    }


def reusable_page_record(
    d,
    key,
    page,
    cohort,
    source_group,
    obs,
    seed_pixels,
):
    """
    Reload just enough saved evidence to reconstruct the
    observable-output summary without rerunning build_arms().
    """

    ambiguous_image = cv2.imread(
        str(d / "ambiguous.png"),
        cv2.IMREAD_GRAYSCALE,
    )

    ring2_image = cv2.imread(
        str(d / "ring2.png"),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        ambiguous_image is not None,
        f"{key}: reusable ambiguous.png unreadable",
    )

    require(
        ring2_image is not None,
        f"{key}: reusable ring2.png unreadable",
    )

    ambiguous = ambiguous_image > 0
    ring2 = ring2_image > 0

    require(
        ambiguous.shape == obs.shape,
        f"{key}: reusable ambiguous shape mismatch",
    )

    require(
        ring2.shape == obs.shape,
        f"{key}: reusable Ring-2 shape mismatch",
    )


    # --------------------------------------------------------
    # Reload the saved truth-free diagnostic arrays.
    # --------------------------------------------------------

    with np.load(
        d / "diagnostic_arrays.npz"
    ) as zz:

        required_arrays = {
            "p_print",
            "pixel_gate",
            "component_gate",
        }

        missing = (
            required_arrays
            - set(zz.files)
        )

        require(
            not missing,
            (
                f"{key}: reusable diagnostic_arrays.npz "
                f"missing {sorted(missing)}"
            ),
        )

        pixel_gate = (
            zz["pixel_gate"] > 0
        )

        component_gate = (
            zz["component_gate"] > 0
        )

        require(
            zz["p_print"].shape == obs.shape,
            f"{key}: reusable p_print shape mismatch",
        )

        require(
            pixel_gate.shape == obs.shape,
            f"{key}: reusable pixel_gate shape mismatch",
        )

        require(
            component_gate.shape == obs.shape,
            f"{key}: reusable component_gate shape mismatch",
        )


    # --------------------------------------------------------
    # Reload the A0-A4 page outputs only long enough to
    # reconstruct changed-pixel counts.
    # --------------------------------------------------------

    changed_counts = {}

    for arm in ARMS:

        arm_path = (
            d / f"{arm}.png"
        )

        img = read_gray(
            arm_path
        )

        require(
            img.shape == obs.shape,
            (
                f"{key}: reusable {arm} "
                "shape mismatch"
            ),
        )

        changed_counts[
            f"{arm}_changed_vs_observed"
        ] = int(
            np.sum(
                img != obs
            )
        )

        del img


    record = {
        "page_key":
            key,

        "page":
            page,

        "cohort":
            cohort,

        "source_group":
            source_group,

        "status":
            "REUSED",

        "skip_reason":
            "",

        "seed_pixels":
            int(seed_pixels),

        "ambiguous_pixels":
            int(ambiguous.sum()),

        "ring2_pixels":
            int(ring2.sum()),

        "pixel_gate_pixels":
            int(pixel_gate.sum()),

        "component_gate_pixels":
            int(component_gate.sum()),

        **changed_counts,
    }


    del (
        ambiguous_image,
        ring2_image,
        ambiguous,
        ring2,
        pixel_gate,
        component_gate,
    )

    gc.collect()

    return record


# ============================================================
# Main resumable processing loop
# ============================================================

output_rows = []


for _, r in roster.iterrows():

    key = str(r.page_key)
    page = str(r.page)
    cohort = str(r.cohort)
    source_group = str(r.source_group)

    d = (
        RUN
        / "pages"
        / key
    )


    # --------------------------------------------------------
    # Load the runtime-observable inputs.
    # --------------------------------------------------------

    obs = read_gray(
        r.observed_path
    )

    seed = read_gray(
        r.seed_path
    )

    require(
        obs.shape == seed.shape,
        (
            f"{key}: "
            "observed/seed mismatch"
        ),
    )

    seed_bool = (
        np.asarray(seed) > 0
    )

    seed_pixels = int(
        seed_bool.sum()
    )


    # ========================================================
    # RESUME PATH
    #
    # Reuse a page only when its entire expected output set
    # already exists.
    # ========================================================

    state = existing_page_state(
        d
    )

    if state == "COMPLETE":

        print(
            f"{key}: REUSE - "
            "complete existing outputs"
        )

        output_rows.append(
            reusable_page_record(
                d=d,
                key=key,
                page=page,
                cohort=cohort,
                source_group=source_group,
                obs=obs,
                seed_pixels=seed_pixels,
            )
        )

        del (
            obs,
            seed,
            seed_bool,
        )

        gc.collect()

        continue


    # --------------------------------------------------------
    # Never silently overwrite or extend an incomplete page.
    #
    # This is intentionally stricter than exist_ok=True.
    # --------------------------------------------------------

    if state == "PARTIAL":

        raise RuntimeError(
            f"{key}: partial existing page output found at:\n"
            f"{d}\n\n"
            "Notebook 81 refuses to mix partial output from "
            "one execution with newly generated output.\n"
            "Inspect that page directory. If the partial "
            "output is known to be invalid, remove that one "
            "page directory explicitly and rerun this cell."
        )


    # ========================================================
    # ZERO-AUTHORITY CONTROL
    # ========================================================

    if seed_pixels == 0:

        print(
            f"{key}: SKIP - "
            "zero-authority page "
            "(reviewed seed contains 0 pixels)"
        )

        output_rows.append(
            skip_record(
                key=key,
                page=page,
                cohort=cohort,
                source_group=source_group,
                status=
                    "SKIPPED_ZERO_AUTHORITY",
                reason=
                    "reviewed_seed_has_zero_pixels",
                seed_pixels=0,
            )
        )

        del (
            obs,
            seed,
            seed_bool,
        )

        gc.collect()

        continue


    # ========================================================
    # NEW PROCESSING PATH
    # ========================================================

    try:

        z = build_arms(
            obs,
            seed,
            page,
        )

    except RuntimeError as e:

        # ----------------------------------------------------
        # Convert ONLY recognised zero-population conditions
        # into an explicit exclusion.
        #
        # Any unrelated failure still aborts the run.
        # ----------------------------------------------------

        msg = str(e).lower()

        known_empty_population = (
            "no m3" in msg
            or "no m3 pixels" in msg
            or "m3 pixels" in msg
            or "no ambiguous" in msg
            or "no ambiguous pixels" in msg
            or "ambiguous pixels" in msg
        )

        if not known_empty_population:
            raise


        print(
            f"{key}: SKIP - "
            "no experimental M3/AMBIGUOUS "
            f"population ({e})"
        )

        output_rows.append(
            skip_record(
                key=key,
                page=page,
                cohort=cohort,
                source_group=source_group,
                status=
                    "SKIPPED_NO_AMBIGUOUS_POPULATION",
                reason=
                    str(e),
                seed_pixels=
                    seed_pixels,
            )
        )

        del (
            obs,
            seed,
            seed_bool,
        )

        gc.collect()

        continue


    # --------------------------------------------------------
    # Defensive check even when build_arms() returned
    # successfully.
    # --------------------------------------------------------

    ambiguous = np.asarray(
        z["ambiguous"],
        dtype=bool,
    )

    ring2 = np.asarray(
        z["ring2"],
        dtype=bool,
    )

    ambiguous_pixels = int(
        ambiguous.sum()
    )

    ring2_pixels = int(
        ring2.sum()
    )


    if ambiguous_pixels == 0:

        print(
            f"{key}: SKIP - "
            "AMBIGUOUS population contains 0 pixels"
        )

        output_rows.append(
            skip_record(
                key=key,
                page=page,
                cohort=cohort,
                source_group=source_group,
                status=
                    "SKIPPED_NO_AMBIGUOUS_POPULATION",
                reason=
                    "ambiguous_population_zero",
                seed_pixels=
                    seed_pixels,
                ambiguous_pixels=
                    0,
                ring2_pixels=
                    ring2_pixels,
            )
        )

        del (
            z,
            ambiguous,
            ring2,
            obs,
            seed,
            seed_bool,
        )

        gc.collect()

        continue


    # ========================================================
    # Persist this newly processed page
    # ========================================================

    d.mkdir(
        parents=True,
        exist_ok=False,
    )


    # --------------------------------------------------------
    # Inputs / route masks
    # --------------------------------------------------------

    require(
        cv2.imwrite(
            str(
                d / "observed.png"
            ),
            obs,
        ),
        f"{key}: failed to write observed.png",
    )

    require(
        cv2.imwrite(
            str(
                d / "seed.png"
            ),
            seed,
        ),
        f"{key}: failed to write seed.png",
    )

    require(
        cv2.imwrite(
            str(
                d / "ambiguous.png"
            ),
            (
                ambiguous.astype(
                    np.uint8
                )
                * 255
            ),
        ),
        f"{key}: failed to write ambiguous.png",
    )

    require(
        cv2.imwrite(
            str(
                d / "ring2.png"
            ),
            (
                ring2.astype(
                    np.uint8
                )
                * 255
            ),
        ),
        f"{key}: failed to write ring2.png",
    )


    # --------------------------------------------------------
    # Truth-free diagnostic arrays
    # --------------------------------------------------------

    np.savez_compressed(
        d / "diagnostic_arrays.npz",

        p_print=
            z["p_print"],

        pixel_gate=
            z["pixel_gate"],

        component_gate=
            z["component_gate"],
    )


    # --------------------------------------------------------
    # A0-A4 arm outputs
    # --------------------------------------------------------

    changed_counts = {}

    for arm, img in z[
        "arms"
    ].items():

        require(
            cv2.imwrite(
                str(
                    d / f"{arm}.png"
                ),
                img,
            ),
            (
                f"{key}: "
                f"failed to write {arm}.png"
            ),
        )

        changed_counts[
            f"{arm}_changed_vs_observed"
        ] = int(
            np.sum(
                img != obs
            )
        )


    # --------------------------------------------------------
    # Record new page
    # --------------------------------------------------------

    output_rows.append({
        "page_key":
            key,

        "page":
            page,

        "cohort":
            cohort,

        "source_group":
            source_group,

        "status":
            "PROCESSED",

        "skip_reason":
            "",

        "seed_pixels":
            seed_pixels,

        "ambiguous_pixels":
            ambiguous_pixels,

        "ring2_pixels":
            ring2_pixels,

        "pixel_gate_pixels":
            int(
                np.asarray(
                    z["pixel_gate"],
                    dtype=bool,
                ).sum()
            ),

        "component_gate_pixels":
            int(
                np.asarray(
                    z["component_gate"],
                    dtype=bool,
                ).sum()
            ),

        **changed_counts,
    })


    print(
        f"{key}: PROCESSED - "
        f"seed={seed_pixels:,}, "
        f"ambiguous={ambiguous_pixels:,}, "
        f"Ring2={ring2_pixels:,}"
    )


    del (
        z,
        ambiguous,
        ring2,
        obs,
        seed,
        seed_bool,
        changed_counts,
    )

    gc.collect()


# ============================================================
# Reconstruct complete run summary from reused + new + skipped
# ============================================================

outputs = pd.DataFrame(
    output_rows
)


require(
    len(outputs) == len(roster),
    (
        "Observable-output summary does not cover the "
        "complete declared roster: "
        f"{len(outputs)} rows for {len(roster)} roster pages."
    ),
)


require(
    not outputs["page_key"].duplicated().any(),
    "Duplicate page_key values in observable-output summary.",
)


outputs.to_csv(
    RUN
    / "tables"
    / "observable_output_summary.csv",

    index=False,
)


# ============================================================
# Processing / exclusion summaries
# ============================================================

usable_status = {
    "PROCESSED",
    "REUSED",
}

processed = outputs[
    outputs[
        "status"
    ].isin(
        usable_status
    )
].copy()

newly_processed = outputs[
    outputs["status"]
    == "PROCESSED"
].copy()

reused = outputs[
    outputs["status"]
    == "REUSED"
].copy()

skipped = outputs[
    ~outputs[
        "status"
    ].isin(
        usable_status
    )
].copy()


# ------------------------------------------------------------
# Human-readable skip summary
# ------------------------------------------------------------

if len(skipped):

    skip_summary = (
        skipped
        .groupby(
            [
                "cohort",
                "status",
                "skip_reason",
            ],
            dropna=False,
        )
        .size()
        .reset_index(
            name="pages"
        )
    )

else:

    skip_summary = pd.DataFrame(
        columns=[
            "cohort",
            "status",
            "skip_reason",
            "pages",
        ]
    )


skip_summary.to_csv(
    RUN
    / "tables"
    / "observable_output_exclusions.csv",

    index=False,
)


print()
print("=" * 72)
print(
    "OBSERVABLE OUTPUT GENERATION COMPLETE"
)
print("=" * 72)

print(
    "Roster pages:",
    len(outputs),
)

print(
    "Experimental pages available:",
    len(processed),
)

print(
    "  newly processed:",
    len(newly_processed),
)

print(
    "  reused:",
    len(reused),
)

print(
    "Skipped zero-exposure pages:",
    len(skipped),
)

print()

if len(skip_summary):

    print(
        skip_summary.to_string(
            index=False
        )
    )


# ============================================================
# Hash EVERY persisted page artefact, whether reused or new
#
# This means a resumed run's identity is based on the final
# filesystem state, not on whether a page happened to be
# processed in the first or second Colab session.
# ============================================================

hash_rows = []


for p in sorted(
    (RUN / "pages").rglob("*")
):

    if not p.is_file():
        continue

    hash_rows.append({
        "path":
            str(
                p.relative_to(
                    RUN
                )
            ),

        "bytes":
            int(
                p.stat().st_size
            ),

        "sha256":
            sha256_file(p),
    })


hashes = pd.DataFrame(
    hash_rows
)


require(
    len(hashes) > 0,
    "No persisted observable output files were found.",
)


hashes.to_csv(
    RUN
    / "audit"
    / "prediction_hashes.csv",

    index=False,
)


# ============================================================
# Hash semantic summary tables
# ============================================================

output_summary_path = (
    RUN
    / "tables"
    / "observable_output_summary.csv"
)

exclusions_path = (
    RUN
    / "tables"
    / "observable_output_exclusions.csv"
)


# ============================================================
# Prediction lock
#
# Important:
#
# "PROCESSED" versus "REUSED" describes computational history.
# It must NOT define experimental identity.
#
# The lock therefore records both counts for audit, while the
# hashes are calculated from the final persisted artefacts.
# ============================================================

lock = {
    "state":
        "CLOSED_BEFORE_CLEAN_TRUTH",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "resume_capable":
        True,

    "roster_pages":
        int(
            len(outputs)
        ),

    "experimental_pages":
        int(
            len(processed)
        ),

    "newly_processed_pages":
        int(
            len(newly_processed)
        ),

    "reused_pages":
        int(
            len(reused)
        ),

    "skipped_pages":
        int(
            len(skipped)
        ),

    "experimental_pages_by_cohort":
        {
            str(k): int(v)
            for k, v in
            processed[
                "cohort"
            ]
            .value_counts()
            .items()
        },

    "skipped_pages_by_cohort":
        {
            str(k): int(v)
            for k, v in
            skipped[
                "cohort"
            ]
            .value_counts()
            .items()
        },

    "skip_reasons":
        skip_summary.to_dict(
            orient="records"
        ),

    "arms":
        list(
            ARMS
        ),

    "pprint_gate":
        PPRINT_GATE,

    # Final persisted file identity.
    "hash_table_sha256":
        hashlib.sha256(
            hashes
            .sort_values(
                "path"
            )
            .to_csv(
                index=False
            )
            .encode()
        ).hexdigest(),

    "observable_output_summary_sha256":
        sha256_file(
            output_summary_path
        ),

    "exclusions_sha256":
        sha256_file(
            exclusions_path
        ),

    "operator_sha256":
        sha256_file(
            V89_OPERATOR_PATH
        ),

    "clean_truth_read":
        False,
}


(
    RUN
    / "prediction_lock.json"
).write_text(
    json.dumps(
        lock,
        indent=2,
    )
)


print()
print("=" * 72)
print("PREDICTION LOCK CLOSED")
print("=" * 72)

print(
    json.dumps(
        lock,
        indent=2,
    )
)

## Post-lock paired truth scoring

Only now are clean pages opened. Primary quantities are calculated **inside AMBIGUOUS coordinates only**, because that is the sole experimental treatment population. A one-pixel tolerant clean-print map is used for genuine-print support. Dark observed pixels with no clean-print support are scored as annotation-like; remaining pixels are truth-uncertain/other and reported separately.


In [ ]:
# ============================================================
# Post-lock paired truth scoring
#
# USER-FACING INTERPRETATION
#
# This cell reports:
#
#   1. AMBIGUOUS_EXPERIMENT
#      What A0-A4 do differently within AMBIGUOUS.
#
#   2. COMMON_CORRECTION
#      What every arm corrects identically outside the
#      experimental AMBIGUOUS treatment.
#
#   3. WHOLE_AUTHORITY
#      The complete user-visible correction result inside the
#      authorised intervention mask.
#
# This distinction is essential:
#
# A1 preserves AMBIGUOUS pixels, so its annotation-improvement
# rate within AMBIGUOUS is expected to be 0%.
#
# A1 can still remove substantial annotation through the
# common REMOVE/paper treatment elsewhere in the authority.
#
# No build_arms() call occurs here. The cell reuses the locked
# outputs written before clean truth was enabled.
# ============================================================


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DARK_THRESHOLD = 200

# Minimum reduction in absolute clean-reference error required
# to call an annotation pixel "materially improved".
#
# This prevents a one-grey-level change from being described
# as meaningful annotation removal.
ANNOTATION_IMPROVEMENT_MARGIN = 5


# ------------------------------------------------------------
# Verify the prediction lock before reading clean truth
# ------------------------------------------------------------

lock = json.loads(
    (
        RUN
        / "prediction_lock.json"
    ).read_text()
)

require(
    lock["state"]
    == "CLOSED_BEFORE_CLEAN_TRUTH",
    "Prediction lock not closed.",
)

require(
    lock.get(
        "clean_truth_read",
        False,
    ) is False,
    (
        "Prediction lock records clean-truth access "
        "before closure."
    ),
)


# ------------------------------------------------------------
# Load the locked observable processing summary.
#
# This is authoritative for determining which paired pages
# actually contain an experimental population.
# ------------------------------------------------------------

observable_summary_path = (
    RUN
    / "tables"
    / "observable_output_summary.csv"
)

require(
    observable_summary_path.exists(),
    (
        "Missing observable_output_summary.csv. "
        "Run the observable-output generation/lock cell first."
    ),
)

observable_summary = pd.read_csv(
    observable_summary_path,
    dtype={
        "page_key": str,
        "page": str,
    },
).fillna("")

require(
    "page_key"
    in observable_summary.columns,
    (
        "observable_output_summary.csv lacks page_key."
    ),
)

require(
    not observable_summary[
        "page_key"
    ].duplicated().any(),
    (
        "Duplicate page_key values in "
        "observable_output_summary.csv."
    ),
)

observable_by_key = (
    observable_summary
    .set_index(
        "page_key",
        drop=False,
    )
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_rate(numerator, denominator):
    """
    Return a numeric rate where a denominator exists,
    otherwise NaN rather than a misleading zero.
    """

    numerator = int(numerator)
    denominator = int(denominator)

    if denominator <= 0:
        return np.nan

    return float(
        numerator / denominator
    )


def score_scope(
    *,
    key,
    page,
    source_group,
    arm,
    scope_name,
    scope_mask,
    obs,
    clean,
    output,
    truth_print,
    truth_annotation,
    truth_uncertain,
):
    """
    Score one arm within one explicit spatial scope.

    The primary user-facing annotation endpoint is movement
    toward the clean reference, not merely a byte difference
    from the observed page.
    """

    scope = np.asarray(
        scope_mask,
        dtype=bool,
    )

    require(
        scope.shape
        == obs.shape
        == clean.shape
        == output.shape,
        (
            f"{key}: score-scope shape mismatch "
            f"for {arm}/{scope_name}."
        ),
    )


    # --------------------------------------------------------
    # Truth populations within this scope
    # --------------------------------------------------------

    print_population = (
        scope
        & truth_print
    )

    annotation_population = (
        scope
        & truth_annotation
    )

    uncertain_population = (
        scope
        & truth_uncertain
    )


    # --------------------------------------------------------
    # Absolute error against the registered clean page
    # --------------------------------------------------------

    obs_i = obs.astype(
        np.int16
    )

    clean_i = clean.astype(
        np.int16
    )

    out_i = output.astype(
        np.int16
    )

    observed_error = np.abs(
        obs_i - clean_i
    )

    output_error = np.abs(
        out_i - clean_i
    )


    # --------------------------------------------------------
    # User-facing annotation outcomes
    #
    # "Materially improved" means the output moved at least
    # ANNOTATION_IMPROVEMENT_MARGIN grey levels closer to clean.
    # --------------------------------------------------------

    annotation_materially_improved = (
        annotation_population
        & (
            output_error
            <= (
                observed_error
                - ANNOTATION_IMPROVEMENT_MARGIN
            )
        )
    )

    annotation_any_improvement = (
        annotation_population
        & (
            output_error
            < observed_error
        )
    )

    annotation_unchanged = (
        annotation_population
        & (
            output_error
            == observed_error
        )
    )

    annotation_worsened = (
        annotation_population
        & (
            output_error
            > observed_error
        )
    )

    annotation_exact_observed = (
        annotation_population
        & (
            output == obs
        )
    )


    # --------------------------------------------------------
    # Genuine-print outcomes
    #
    # Exact observed-byte preservation remains important
    # because that is the frozen observed-print safety action.
    #
    # Clean-error outcomes are also reported, because an
    # observed print pixel may itself be contaminated by
    # overlapping annotation.
    # --------------------------------------------------------

    print_exact_observed = (
        print_population
        & (
            output == obs
        )
    )

    print_materially_improved = (
        print_population
        & (
            output_error
            <= (
                observed_error
                - ANNOTATION_IMPROVEMENT_MARGIN
            )
        )
    )

    print_not_worsened = (
        print_population
        & (
            output_error
            <= observed_error
        )
    )

    print_worsened = (
        print_population
        & (
            output_error
            > observed_error
        )
    )

    print_materially_worsened = (
        print_population
        & (
            output_error
            >= (
                observed_error
                + ANNOTATION_IMPROVEMENT_MARGIN
            )
        )
    )


    # --------------------------------------------------------
    # Mean absolute errors for readable effect size
    # --------------------------------------------------------

    annotation_obs_mae = (
        float(
            observed_error[
                annotation_population
            ].mean()
        )
        if annotation_population.any()
        else np.nan
    )

    annotation_output_mae = (
        float(
            output_error[
                annotation_population
            ].mean()
        )
        if annotation_population.any()
        else np.nan
    )

    print_obs_mae = (
        float(
            observed_error[
                print_population
            ].mean()
        )
        if print_population.any()
        else np.nan
    )

    print_output_mae = (
        float(
            output_error[
                print_population
            ].mean()
        )
        if print_population.any()
        else np.nan
    )


    print_pixels = int(
        print_population.sum()
    )

    annotation_pixels = int(
        annotation_population.sum()
    )

    uncertain_pixels = int(
        uncertain_population.sum()
    )

    annotation_improved_pixels = int(
        annotation_materially_improved.sum()
    )

    print_preserved_pixels = int(
        print_exact_observed.sum()
    )

    return {
        "page_key":
            key,

        "page":
            page,

        "source_group":
            source_group,

        "arm":
            arm,

        "scope":
            scope_name,

        "scope_pixels":
            int(
                scope.sum()
            ),

        "truth_print_pixels":
            print_pixels,

        "truth_annotation_pixels":
            annotation_pixels,

        "truth_uncertain_pixels":
            uncertain_pixels,


        # ----------------------------------------------------
        # User-facing annotation fields
        # ----------------------------------------------------

        "annotation_materially_improved_toward_clean":
            annotation_improved_pixels,

        "annotation_any_improvement_toward_clean":
            int(
                annotation_any_improvement.sum()
            ),

        "annotation_unchanged_vs_clean_error":
            int(
                annotation_unchanged.sum()
            ),

        "annotation_worsened_vs_clean_error":
            int(
                annotation_worsened.sum()
            ),

        "annotation_left_exactly_observed":
            int(
                annotation_exact_observed.sum()
            ),

        "annotation_material_improvement_rate":
            safe_rate(
                annotation_improved_pixels,
                annotation_pixels,
            ),

        "annotation_any_improvement_rate":
            safe_rate(
                annotation_any_improvement.sum(),
                annotation_pixels,
            ),

        "annotation_observed_mae":
            annotation_obs_mae,

        "annotation_output_mae":
            annotation_output_mae,

        "annotation_mae_improvement":
            (
                annotation_obs_mae
                - annotation_output_mae
                if np.isfinite(
                    annotation_obs_mae
                )
                and np.isfinite(
                    annotation_output_mae
                )
                else np.nan
            ),


        # ----------------------------------------------------
        # User-facing genuine-print fields
        # ----------------------------------------------------

        "print_preserved_exact_observed":
            print_preserved_pixels,

        "print_materially_improved_toward_clean":
            int(
                print_materially_improved.sum()
            ),

        "print_not_worsened_vs_clean_error":
            int(
                print_not_worsened.sum()
            ),

        "print_worsened_vs_clean_error":
            int(
                print_worsened.sum()
            ),

        "print_materially_worsened_vs_clean":
            int(
                print_materially_worsened.sum()
            ),

        "print_exact_observed_preservation_rate":
            safe_rate(
                print_preserved_pixels,
                print_pixels,
            ),

        "print_not_worsened_rate":
            safe_rate(
                print_not_worsened.sum(),
                print_pixels,
            ),

        "print_material_worsening_rate":
            safe_rate(
                print_materially_worsened.sum(),
                print_pixels,
            ),

        "print_observed_mae":
            print_obs_mae,

        "print_output_mae":
            print_output_mae,

        "print_mae_change":
            (
                print_output_mae
                - print_obs_mae
                if np.isfinite(
                    print_obs_mae
                )
                and np.isfinite(
                    print_output_mae
                )
                else np.nan
            ),
    }


# ============================================================
# Score paired pages
# ============================================================

score_rows = []
pixel_parts = []
skip_rows = []


paired = roster[
    roster["cohort"]
    == "PAIRED_DEVELOPMENT"
].copy()


for _, r in paired.iterrows():

    key = str(
        r.page_key
    )

    page = str(
        r.page
    )

    source_group = str(
        r.source_group
    )


    # --------------------------------------------------------
    # Respect the locked observable exclusion decision.
    # --------------------------------------------------------

    if key not in observable_by_key.index:

        raise RuntimeError(
            f"{key}: page absent from locked "
            "observable_output_summary.csv."
        )


    status = str(
        observable_by_key.loc[
            key,
            "status",
        ]
    )


    if status not in {
        "PROCESSED",
        "REUSED",
    }:

        reason = str(
            observable_by_key.loc[
                key,
                "skip_reason",
            ]
        )

        print(
            f"{key}: SKIP scoring - "
            f"{status}: {reason}"
        )

        skip_rows.append({
            "page_key":
                key,

            "page":
                page,

            "source_group":
                source_group,

            "status":
                status,

            "skip_reason":
                reason,
        })

        continue


    # --------------------------------------------------------
    # Resolve the locked outputs.
    # --------------------------------------------------------

    d = (
        RUN
        / "pages"
        / key
    )

    require(
        d.is_dir(),
        (
            f"{key}: locked page-output "
            f"directory missing: {d}"
        ),
    )


    obs = read_gray(
        r.observed_path
    )

    clean = read_gray(
        r.clean_path
    )

    seed = read_gray(
        r.seed_path
    )


    require(
        obs.shape
        == clean.shape
        == seed.shape,
        f"{key}: observed/clean/seed shape mismatch",
    )


    # --------------------------------------------------------
    # Load the saved experimental masks.
    #
    # No build_arms() call is made here.
    # --------------------------------------------------------

    ambiguous = (
        read_gray(
            d / "ambiguous.png"
        ) > 0
    )

    ring2 = (
        read_gray(
            d / "ring2.png"
        ) > 0
    )


    require(
        ambiguous.shape
        == obs.shape,
        f"{key}: ambiguous-mask shape mismatch",
    )

    if not ambiguous.any():

        print(
            f"{key} SKIP scoring - "
            "zero AMBIGUOUS pixels; "
            "no A0-A4 experimental exposure."
        )

        skip_rows.append({
            "page_key": key,
            "page": page,
            "source_group": source_group,
            "status": "SKIPPED_ZERO_AMBIGUOUS",
            "skip_reason": (
                "authoritative mask is non-empty, "
                "but frozen routing contains zero "
                "AMBIGUOUS pixels"
            ),
        })

        del (
            obs,
            clean,
            seed,
            ambiguous,
            ring2,
        )

        gc.collect()

        continue



    


    # --------------------------------------------------------
    # Recreate the authoritative prepared mask from the
    # reviewed seed using the frozen dilation-2 contract.
    # --------------------------------------------------------

    seed_bool = (
        seed > 0
    )

    k2 = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5),
    )

    authority = (
        cv2.dilate(
            seed_bool.astype(
                np.uint8
            ),
            k2,
        ) > 0
    )


    # --------------------------------------------------------
    # Clean-truth maps
    # --------------------------------------------------------

    clean_print = printed_ink(
        clean
    )

    clean_tol = (
        cv2.dilate(
            clean_print.astype(
                np.uint8
            ),
            np.ones(
                (3, 3),
                np.uint8,
            ),
        ) > 0
    )


    # Genuine printed material anywhere in authority.
    truth_print = (
        authority
        & clean_tol
    )


    # Dark observed material without tolerant clean-print
    # support is treated as truth-side annotation.
    truth_annotation = (
        authority
        & (
            obs < DARK_THRESHOLD
        )
        & ~clean_tol
    )


    # Keep all remaining authority pixels separate.
    truth_uncertain = (
        authority
        & ~truth_print
        & ~truth_annotation
    )


    # --------------------------------------------------------
    # Load all locked A0-A4 outputs.
    # --------------------------------------------------------

    arm_images = {
        arm:
            read_gray(
                d / f"{arm}.png"
            )

        for arm in ARMS
    }


    for arm, img in arm_images.items():

        require(
            img.shape
            == obs.shape,
            (
                f"{key}: {arm} "
                "output shape mismatch."
            ),
        )


    # --------------------------------------------------------
    # Common correction:
    #
    # Pixels changed from observed identically by every arm.
    #
    # This captures the shared REMOVE/paper and any other
    # non-experimental treatment without pretending to infer
    # route names from the final images.
    # --------------------------------------------------------

    first_arm = ARMS[0]

    common_correction = (
        authority
        & (
            arm_images[
                first_arm
            ] != obs
        )
    )


    for arm in ARMS[1:]:

        common_correction &= (
            arm_images[arm]
            == arm_images[
                first_arm
            ]
        )


    # --------------------------------------------------------
    # Score every arm in three explicit scopes.
    # --------------------------------------------------------

    for arm, img in arm_images.items():

        score_rows.append(
            score_scope(
                key=key,
                page=page,
                source_group=source_group,
                arm=arm,
                scope_name=
                    "AMBIGUOUS_EXPERIMENT",
                scope_mask=
                    ambiguous,
                obs=obs,
                clean=clean,
                output=img,
                truth_print=
                    truth_print,
                truth_annotation=
                    truth_annotation,
                truth_uncertain=
                    truth_uncertain,
            )
        )

        score_rows.append(
            score_scope(
                key=key,
                page=page,
                source_group=source_group,
                arm=arm,
                scope_name=
                    "COMMON_CORRECTION",
                scope_mask=
                    common_correction,
                obs=obs,
                clean=clean,
                output=img,
                truth_print=
                    truth_print,
                truth_annotation=
                    truth_annotation,
                truth_uncertain=
                    truth_uncertain,
            )
        )

        score_rows.append(
            score_scope(
                key=key,
                page=page,
                source_group=source_group,
                arm=arm,
                scope_name=
                    "WHOLE_AUTHORITY",
                scope_mask=
                    authority,
                obs=obs,
                clean=clean,
                output=img,
                truth_print=
                    truth_print,
                truth_annotation=
                    truth_annotation,
                truth_uncertain=
                    truth_uncertain,
            )
        )


    # --------------------------------------------------------
    # Pixel-level AMBIGUOUS table.
    #
    # This remains useful for analysing the A0-A4 experimental
    # mechanism, but is explicitly labelled AMBIGUOUS-only.
    # --------------------------------------------------------

    ys, xs = np.where(
        ambiguous
    )

    if len(ys):

        part = pd.DataFrame({
            "page_key":
                key,

            "page":
                page,

            "source_group":
                source_group,

            "y":
                ys,

            "x":
                xs,

            "truth_print":
                truth_print[
                    ys,
                    xs,
                ],

            "truth_annotation":
                truth_annotation[
                    ys,
                    xs,
                ],

            "truth_uncertain":
                truth_uncertain[
                    ys,
                    xs,
                ],
        })


        # p_print is truth-free and was locked before clean.
        with np.load(
            d
            / "diagnostic_arrays.npz"
        ) as zz:

            require(
                "p_print"
                in zz.files,
                (
                    f"{key}: diagnostic arrays "
                    "lack p_print."
                ),
            )

            part["p_print"] = (
                zz["p_print"][
                    ys,
                    xs,
                ]
            )


        for arm, img in arm_images.items():

            output_error = np.abs(
                img.astype(
                    np.int16
                )
                - clean.astype(
                    np.int16
                )
            )

            observed_error = np.abs(
                obs.astype(
                    np.int16
                )
                - clean.astype(
                    np.int16
                )
            )

            part[
                f"{arm}_equals_observed"
            ] = (
                img[
                    ys,
                    xs,
                ]
                == obs[
                    ys,
                    xs,
                ]
            )

            part[
                f"{arm}_materially_closer_to_clean"
            ] = (
                output_error[
                    ys,
                    xs,
                ]
                <= (
                    observed_error[
                        ys,
                        xs,
                    ]
                    - ANNOTATION_IMPROVEMENT_MARGIN
                )
            )

            part[
                f"{arm}_materially_farther_from_clean"
            ] = (
                output_error[
                    ys,
                    xs,
                ]
                >= (
                    observed_error[
                        ys,
                        xs,
                    ]
                    + ANNOTATION_IMPROVEMENT_MARGIN
                )
            )


        pixel_parts.append(
            part
        )


    print(
        f"{key}: SCORED - "
        f"authority={int(authority.sum()):,}, "
        f"ambiguous={int(ambiguous.sum()):,}, "
        f"common={int(common_correction.sum()):,}"
    )


    del (
        obs,
        clean,
        seed,
        seed_bool,
        ambiguous,
        ring2,
        authority,
        clean_print,
        clean_tol,
        truth_print,
        truth_annotation,
        truth_uncertain,
        common_correction,
        arm_images,
    )

    gc.collect()


# ============================================================
# Save detailed outputs
# ============================================================

scores = pd.DataFrame(
    score_rows
)

require(
    len(scores) > 0,
    "No paired pages were scored.",
)


scores.to_csv(
    RUN
    / "tables"
    / "paired_truth_by_page_and_scope.csv",

    index=False,
)


skips = pd.DataFrame(
    skip_rows
)

skips.to_csv(
    RUN
    / "tables"
    / "paired_truth_scoring_exclusions.csv",

    index=False,
)


pixels = (
    pd.concat(
        pixel_parts,
        ignore_index=True,
    )
    if pixel_parts
    else pd.DataFrame()
)

pixels.to_parquet(
    RUN
    / "tables"
    / "paired_ambiguous_pixel_truth.parquet",

    index=False,
)


# ============================================================
# Pooled user-facing summary
# ============================================================

sum_columns = [
    "scope_pixels",

    "truth_print_pixels",
    "truth_annotation_pixels",
    "truth_uncertain_pixels",

    "annotation_materially_improved_toward_clean",
    "annotation_any_improvement_toward_clean",
    "annotation_unchanged_vs_clean_error",
    "annotation_worsened_vs_clean_error",
    "annotation_left_exactly_observed",

    "print_preserved_exact_observed",
    "print_materially_improved_toward_clean",
    "print_not_worsened_vs_clean_error",
    "print_worsened_vs_clean_error",
    "print_materially_worsened_vs_clean",
]


pooled = (
    scores
    .groupby(
        [
            "scope",
            "arm",
        ],
        as_index=False,
    )
    .agg(
        pages=(
            "page_key",
            "nunique",
        ),

        **{
            column: (
                column,
                "sum",
            )
            for column
            in sum_columns
        },

        annotation_observed_absolute_error=(
            "annotation_observed_mae",
            lambda q: np.nan,
        ),
    )
)


# Drop the placeholder aggregate. Pooled MAE should be produced
# from pixel totals if later required; page-mean MAE averaging
# would otherwise weight small and large pages equally.
pooled = pooled.drop(
    columns=[
        "annotation_observed_absolute_error"
    ]
)


# ------------------------------------------------------------
# Derive pooled rates explicitly
# ------------------------------------------------------------

pooled[
    "annotation_material_improvement_rate"
] = (
    pooled[
        "annotation_materially_improved_toward_clean"
    ]
    /
    pooled[
        "truth_annotation_pixels"
    ].clip(
        lower=1
    )
)

pooled[
    "annotation_any_improvement_rate"
] = (
    pooled[
        "annotation_any_improvement_toward_clean"
    ]
    /
    pooled[
        "truth_annotation_pixels"
    ].clip(
        lower=1
    )
)

pooled[
    "annotation_left_observed_rate"
] = (
    pooled[
        "annotation_left_exactly_observed"
    ]
    /
    pooled[
        "truth_annotation_pixels"
    ].clip(
        lower=1
    )
)

pooled[
    "print_exact_observed_preservation_rate"
] = (
    pooled[
        "print_preserved_exact_observed"
    ]
    /
    pooled[
        "truth_print_pixels"
    ].clip(
        lower=1
    )
)

pooled[
    "print_not_worsened_rate"
] = (
    pooled[
        "print_not_worsened_vs_clean_error"
    ]
    /
    pooled[
        "truth_print_pixels"
    ].clip(
        lower=1
    )
)

pooled[
    "print_material_worsening_rate"
] = (
    pooled[
        "print_materially_worsened_vs_clean"
    ]
    /
    pooled[
        "truth_print_pixels"
    ].clip(
        lower=1
    )
)


pooled.to_csv(
    RUN
    / "tables"
    / "pooled_arm_summary_by_scope.csv",

    index=False,
)


# ============================================================
# Compact user-facing view
# ============================================================

user_view_columns = [
    "scope",
    "arm",
    "pages",

    "truth_print_pixels",
    "truth_annotation_pixels",

    "print_exact_observed_preservation_rate",
    "print_not_worsened_rate",
    "print_material_worsening_rate",

    "annotation_material_improvement_rate",
    "annotation_any_improvement_rate",
    "annotation_left_observed_rate",
]


user_view = pooled[
    user_view_columns
].copy()


display(
    user_view.sort_values(
        [
            "scope",
            "arm",
        ]
    )
)


# ============================================================
# Explicit interpretation note
# ============================================================

print()
print("=" * 72)
print("HOW TO READ THESE RESULTS")
print("=" * 72)

print(
    "AMBIGUOUS_EXPERIMENT isolates the treatment changed by "
    "A0-A4."
)

print(
    "For A1, annotation improvement within AMBIGUOUS should "
    "normally be zero because A1 copies the observed byte there."
)

print(
    "COMMON_CORRECTION captures annotation removed identically "
    "by every arm, principally the shared non-AMBIGUOUS "
    "correction path."
)

print(
    "WHOLE_AUTHORITY is the user-facing end-to-end result and "
    "is the correct scope for statements such as 'how much "
    "annotation did the completed page remove?'"
)

print()
print(
    "Scored paired pages:",
    int(
        scores[
            "page_key"
        ].nunique()
    ),
)

print(
    "Skipped paired pages:",
    len(skips),
)

## Visual summaries

The first chart deliberately treats print preservation and annotation removal as separate axes. There is no single utility score. The second chart shows page-level trade-offs. Contact sheets use selected paired pages and HITL pages with the largest A0↔A1 visual difference.


In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
for _,r in pooled.iterrows():
    ax.scatter(r.annotation_removal_rate,r.print_preservation_rate,s=80,label=r.arm)
    ax.text(r.annotation_removal_rate,r.print_preservation_rate,r.arm.replace('_',' '),fontsize=8)
ax.set_xlabel('Annotation removal rate within AMBIGUOUS truth-annotation pixels')
ax.set_ylabel('Genuine print preservation rate within AMBIGUOUS truth-print pixels')
ax.set_xlim(0,1.02);ax.set_ylim(0,1.02);ax.grid(True,alpha=.25);plt.tight_layout();plt.savefig(RUN/'visual'/'arm_tradeoff.png',dpi=160);plt.close(fig)

trade=scores.pivot(index='page_key',columns='arm',values=['print_preserved_observed','annotation_retained_observed']).reset_index()
# Flatten for easy downstream reading.
trade.columns=['__'.join([str(x) for x in c if str(x)]) if isinstance(c,tuple) else str(c) for c in trade.columns]
trade.to_csv(RUN/'tables'/'page_tradeoff_wide.csv',index=False)

obs_summary=pd.read_csv(RUN/'tables'/'observable_output_summary.csv')
obs_summary['A0_A1_delta']=0
for i,r in obs_summary.iterrows():
    d=RUN/'pages'/str(r.page_key);a0=read_gray(d/'A0_TELEA.png');a1=read_gray(d/'A1_OBSERVED.png');obs_summary.loc[i,'A0_A1_delta']=int(np.sum(a0!=a1))
obs_summary=obs_summary.sort_values('A0_A1_delta',ascending=False)
obs_summary.to_csv(RUN/'tables'/'pages_ranked_by_A0_A1_difference.csv',index=False)

def make_contacts(frame,name,n=24,per=6):
    qs=frame.head(n)
    paths=[]
    for start in range(0,len(qs),per):
        q=qs.iloc[start:start+per];fig,axes=plt.subplots(len(q),6,figsize=(24,max(5,4.7*len(q))),squeeze=False)
        for rr,(_,row) in enumerate(q.iterrows()):
            d=RUN/'pages'/str(row.page_key)
            ips=[d/'observed.png']+[d/(a+'.png') for a in ARMS]
            titles=['Observed']+ARMS
            for cc,(p,t) in enumerate(zip(ips,titles)):
                im=read_gray(p);axes[rr,cc].imshow(im,cmap='gray',vmin=0,vmax=255);axes[rr,cc].set_title((str(row.page)+'\n'+t) if cc==0 else t,fontsize=7);axes[rr,cc].axis('off')
        plt.tight_layout();p=RUN/'visual'/f'{name}_{start//per+1:02d}.png';plt.savefig(p,dpi=105,bbox_inches='tight');plt.close(fig);paths.append(p)
    return paths
paired_rank=obs_summary[obs_summary.cohort=='PAIRED_DEVELOPMENT'];hitl_rank=obs_summary[obs_summary.cohort=='HITL_ANNOTATED_ONLY']
contacts=make_contacts(paired_rank,'paired_contact')+make_contacts(hitl_rank,'hitl_contact')
print('Contact sheets:',len(contacts))


## Final diagnostic verdict and upload bundle

This notebook does **not** promote an arm. It reports the Pareto-style trade-off. A later decision can favour retained musical information over cosmetic cleanliness, consistent with the visual-restorer objective.


In [ ]:
pooled=pd.read_csv(RUN/'tables'/'pooled_arm_summary.csv')
verdict={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'notebook':'81_AMBIGUOUS_Treatment_Benchmark',
    'status':'DIAGNOSTIC_COMPLETE_NO_AUTOMATIC_WINNER',
    'arms':ARMS,
    'paired_pages':int(roster[roster.cohort=='PAIRED_DEVELOPMENT'].shape[0]),
    'hitl_pages':int(roster[roster.cohort=='HITL_ANNOTATED_ONLY'].shape[0]),
    'prediction_lock':'CLOSED_BEFORE_CLEAN_TRUTH',
    'uses_clean_truth_for_inference':False,
    'promotion_authority':False,
    'interpretation':'Compare print preservation and annotation removal separately; do not collapse them into one utility score.'
}
(RUN/'notebook81_verdict.json').write_text(json.dumps(verdict,indent=2))
summary=[
    'NOTEBOOK 81 - AMBIGUOUS TREATMENT BENCHMARK',
    '',
    f'HITL pages: {verdict["hitl_pages"]}',
    f'Paired pages: {verdict["paired_pages"]}',
    '',
    'Primary output: pooled_arm_summary.csv',
    'Visual output: arm_tradeoff.png plus paired/HITL contact sheets',
    '',
    'No production promotion is authorised.'
]
summary_text='\n'.join(summary)
(RUN/'executive_summary.txt').write_text(summary_text)
for pth in [
    RUN/'executive_summary.txt',
    RUN/'notebook81_verdict.json',
    RUN/'prediction_lock.json',
    RUN/'tables'/'pooled_arm_summary.csv',
    RUN/'tables'/'paired_truth_by_page.csv',
    RUN/'tables'/'pages_ranked_by_A0_A1_difference.csv',
    RUN/'visual'/'arm_tradeoff.png',
    *contacts,
]:
    shutil.copy2(pth,RUN/'upload'/pth.name)
shutil.make_archive(str(RUN/'notebook81_ambiguous_treatment_bundle'),'zip',RUN/'upload')
print(summary_text)
display(pooled)
print('COMPLETE:',RUN)
